# tsfresh feature generation and t-sne analysis on StressID and ExpData datasets

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from dataclasses import dataclass
from typing import Tuple, TypeAlias

from tsfresh import extract_features, select_features

from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, SimpleImputer

BASE_PATH = "../../.."
DATASET = f"{BASE_PATH}/stressid-dataset"
DATA_ECG = f"{DATASET}/ecg_windowed.csv"
DATA_EDA = f"{DATASET}/eda_windowed.csv"
LABELS_SEPARATOR = ","
LABELS = f"{BASE_PATH}/stressID/labels.csv"
DATA_SEPARATOR = ","
DATA_FS = 500  # Hz
DATA_WINDOW_DURATION = 60  # seconds
TARGET_FS = 51.2
RANDOM_STATE = 21

BIN_LABELS = ["NoStress", "Stress"]
TER_LABELS = ["Relaxed", "Stress", "RealStress"]
QAD_LABELS = ["Relaxed", "Stress", "RealStress", "Amused"]

LABELS_CONF = {
    "b": {
        "col_name": "binary-stress",
        "enabled": True,
        "stratification": True,
        "classes": BIN_LABELS,
    },
    "t": {
        "col_name": "affect3-class",
        "enabled": False,
        "stratification": True,
        "classes": TER_LABELS,
    },
    "q": {
        "col_name": "affect4-class",
        "enabled": False,
        "stratification": True,
        "classes": QAD_LABELS,
    },
}


@dataclass
class Dataset:
    X: list[pd.Series]
    y: pd.Series
    groups: np.ndarray[int]


CWT: TypeAlias = Tuple[np.ndarray[tuple[int], np.dtype], np.ndarray]

### Build dataset from data files

In [ ]:
# Creating labels object

labels_df = pd.read_csv(LABELS, sep=LABELS_SEPARATOR, header=0, index_col=0)
labels: dict[str, pd.Series] = {}
for key, conf in LABELS_CONF.items():
    if conf["enabled"]:
        labels[key] = labels_df[conf["col_name"]]

display(labels["b"])


In [ ]:
# Pairing labels and samples for each class type

raw_num_samples = DATA_FS * DATA_WINDOW_DURATION
raw_eda = pd.read_csv(DATA_EDA)
raw_ecg = pd.read_csv(DATA_ECG)

subject_series: int = []  # "Subject"
task_series: str = []  # "Task"
sample_n_series: int = []  # "Nth-Sample"
eda_series: float = []  # "EDA"
ecg_series: float = []  # "ECG"

blabel_series: int = []  # "b-class"
tlabel_series: int = []  # "t-class"
qlabel_series: int = []  # "q-class"

for class_type, labels_set in labels.items():
    subject_to_group: dict[str, int] = {}
    group_counter = 0
    groups_list: list[int] = []

    for col_name, label in labels_set.items():
        subject_id = col_name.split("_")[0]
        if (
            col_name in raw_eda.columns
        ):  # Only add labels with corresponding data, EDA dataset lacks one entry which ECG has
            length = raw_eda[col_name].size
            sample_n_series.extend([n for n in range(length)])
            eda_series.extend(raw_eda[col_name])
            ecg_series.extend(raw_ecg[col_name])
            label_values = np.full(length, label)
            if class_type == "b":
                blabel_series.extend(label_values)
            elif class_type == "t":
                tlabel_series.extend(label_values)
            elif class_type == "q":
                qlabel_series.extend(label_values)
            if subject_id not in subject_to_group:
                subject_to_group[subject_id] = group_counter
                group_counter += 1
            subject_series.extend(np.full(length, subject_to_group[subject_id]))
            task_series.extend(np.full(length, col_name))

data_df = pd.DataFrame({
    # "Subject": subject_series,
    "Task": task_series,
    "Nth-Sample": sample_n_series,
    "EDA": eda_series,
    "ECG": ecg_series})
display(data_df)

In [ ]:
features = extract_features(data_df, column_id="Task", column_sort="Nth-Sample", impute_function=SimpleImputer)
